# RagFlow接口测试

In [1]:
import requests

In [10]:
ADDRESS = "http://192.168.111.200:9380"
CHAT_ID = 'dc87126e504611f19aceb391bf03f413'
API_KEY = "ragflow-L-Uf9Mrlk7IsJWgS25O5keEqraqudBwUWYfaeuPoSZc"

### 创建会话

In [11]:
url = f"{ADDRESS}/api/v1/chats/{CHAT_ID}/sessions"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}",
}

payload = {
    "name": "接口测试"
}

In [16]:
response = requests.post(
    url=url,
    headers=headers,
    json=payload,
    timeout=30,
)
print("HTTP 状态码:", response.status_code)

data = response.json()
print("响应 JSON:", data)


HTTP 状态码: 200
响应 JSON: {'code': 0, 'data': {'chat_id': 'dc87126e504611f19aceb391bf03f413', 'create_date': '2026-05-17T12:37:08', 'create_time': 1778992628824, 'id': '107f022a51aa11f1a5be253d403d7ec5', 'messages': [{'content': '你好！ 我是你的助理，有什么可以帮到你的吗？', 'role': 'assistant'}], 'name': '接口测试', 'update_date': '2026-05-17T12:37:08', 'update_time': 1778992628824, 'user_id': ''}}


In [17]:
data

{'code': 0,
 'data': {'chat_id': 'dc87126e504611f19aceb391bf03f413',
  'create_date': '2026-05-17T12:37:08',
  'create_time': 1778992628824,
  'id': '107f022a51aa11f1a5be253d403d7ec5',
  'messages': [{'content': '你好！ 我是你的助理，有什么可以帮到你的吗？', 'role': 'assistant'}],
  'name': '接口测试',
  'update_date': '2026-05-17T12:37:08',
  'update_time': 1778992628824,
  'user_id': ''}}

## 流式输出测试

curl --request POST \
     --url http://{address}/api/v1/chats/{chat_id}/completions \
     --header 'Content-Type: application/json' \
     --header 'Authorization: Bearer <YOUR_API_KEY>' \
     --data-binary '
     {
          "question": "Who are you",
          "stream": true,
          "session_id":"9fa7691cb85c11ef9c5f0242ac120005",
          "metadata_condition": {
            "logic": "and",
            "conditions": [
              {
                "name": "author",
                "comparison_operator": "is",
                "value": "bob"
              }
            ]
          }
     }'

In [30]:
ADDRESS = "http://192.168.111.200"
CHAT_ID = 'dc87126e504611f19aceb391bf03f413'
API_KEY = "ragflow-L-Uf9Mrlk7IsJWgS25O5keEqraqudBwUWYfaeuPoSZc"
SESSION_ID = "107f022a51aa11f1a5be253d403d7ec5"

In [25]:
url = f"{ADDRESS}/api/v1/chats/{CHAT_ID}/completions"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}",
}

payload = {
    "question": "Who are you",
    "stream": 'true',
    "session_id":SESSION_ID,
    "metadata_condition": {
    "logic": "and",
    "conditions": [
        {
        "name": "author",
        "comparison_operator": "is",
        "value": "bob"
        }
    ]
    }
}

In [28]:
import json
import time
import requests

In [32]:
import json
import requests


# ADDRESS = "http://127.0.0.1:9380"
# API_KEY = "你的_API_KEY"
# CHAT_ID = "你的_CHAT_ID"
# SESSION_ID = "你的_SESSION_ID"


def main():
    url = f"{ADDRESS.rstrip('/')}/api/v1/chats/{CHAT_ID}/completions"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }

    payload = {
        "question": "城市更新主要任务是什么？",
        "stream": True,
        "session_id": SESSION_ID,
    }

    print("请求地址:", url)
    print("请求参数:")
    print(json.dumps(payload, ensure_ascii=False, indent=2))
    print("=" * 80)

    last_answer = ""

    with requests.post(
        url,
        headers=headers,
        json=payload,
        stream=True,
        timeout=(10, 180),
    ) as response:
        print("HTTP 状态码:", response.status_code)
        print("=" * 80)

        if response.status_code != 200:
            print(response.text)
            return

        for line in response.iter_lines(decode_unicode=True):
            if not line:
                continue

            if line.startswith("data:"):
                line = line[len("data:"):].strip()

            if not line:
                continue

            obj = json.loads(line)

            # 流式结束标志
            if obj.get("data") is True:
                print("\n\n[DONE]")
                break

            data = obj.get("data", {})
            answer = data.get("answer", "")

            # RAGFlow 返回的 answer 通常是累计文本，这里只打印新增部分
            if answer:
                if answer.startswith(last_answer):
                    delta = answer[len(last_answer):]
                else:
                    delta = answer

                print(delta, end="", flush=True)
                last_answer = answer

            # 如果有引用，简单打印一下命中文档
            reference = data.get("reference")
            if reference and reference.get("chunks"):
                print("\n\n[命中知识库]")
                for chunk in reference["chunks"]:
                    print("-", chunk.get("document_name"), "similarity:", chunk.get("similarity"))


if __name__ == "__main__":
    main()

请求地址: http://192.168.111.200/api/v1/chats/dc87126e504611f19aceb391bf03f413/completions
请求参数:
{
  "question": "城市更新主要任务是什么？",
  "stream": true,
  "session_id": "107f022a51aa11f1a5be253d403d7ec5"
}
HTTP 状态码: 200
嗯，用户问的是“城市更新主要任务是什么？”，我需要从知识库里找到相关信息来回答。首先，我浏览知识库，发现ID:0和ID:3的内容提到了城市更新的主要任务。ID:0的正文部分明确列出了八项主要任务，包括加强既有建筑改造、推进老旧小区整治、社区建设等。ID:3的切片ID:3也详细说明了这些任务，围绕人民的需求来实施。其他知识库条目主要涉及规划编制和实施机制，但用户的问题更关注任务本身，所以重点放在ID:0和ID:3的内容上。我应该将这些任务逐一列出，并标注相应的引用来源，确保每个任务都有对应的知识库支持。同时，要保持回答的结构化，使用清晰的分点形式，让用户容易理解。


城市更新的主要任务包括：

1. **加强既有建筑改造利用**  
2. **推进城镇老旧小区整治改造**  
3. **开展完整社区建设**  
4. **推进老旧街区、老旧厂区、城中村等更新改造**  
5. **完善城市功能**  
6. **加强城市基础设施建设改造**  
7. **修复城市生态系统**  
8. **保护传承城市历史文化**[ID:0][ID:3]。

这些任务紧密围绕新时期人民需求，立足关键领域精准发力[ID:3]。

[命中知识库]
- 中央明确城市更新“路线图”！宜居、韧性、智慧_20260313.md similarity: 0.5427810629358258
- 节能处-01城市更新-参阅材料-2022-【政策文件】各省城市更新文件-【江西城市更新规划】-附件：江西省城市更新规划编制指南（试行）.md similarity: 0.5375082623780895
- 节能处-01城市更新-参阅材料-2022-【政策文件】各省城市更新文件-【江西城市更新规划】-附件：江西省城市更新规划编制指南（试行）.md similarity: 0.535937863

# 项目测试

In [1]:
from app.core.config import settings


def mask(value: str, keep: int = 6) -> str:
    if not value:
        return ""
    return value[:keep] + "******"


print("APP_NAME =", settings.APP_NAME)
print("RAGFLOW_BASE_URL =", settings.RAGFLOW_BASE_URL)
print("RAGFLOW_API_KEY =", mask(settings.RAGFLOW_API_KEY))

print("OFFICE KB =", settings.RAGFLOW_OFFICE_LEADER_KB_CHAT_ID)
print("OFFICE SUMMARY =", settings.RAGFLOW_OFFICE_LEADER_SUMMARY_CHAT_ID)
print("DEPARTMENT KB =", settings.RAGFLOW_DEPARTMENT_LEADER_KB_CHAT_ID)
print("DEPARTMENT SUMMARY =", settings.RAGFLOW_DEPARTMENT_LEADER_SUMMARY_CHAT_ID)

print("GLOBAL_CONCURRENCY =", settings.GLOBAL_CONCURRENCY)
print("RAGFLOW_CONCURRENCY =", settings.RAGFLOW_CONCURRENCY)

APP_NAME = StreamGate
RAGFLOW_BASE_URL = http://192.168.111.200
RAGFLOW_API_KEY = ragflo******
OFFICE KB = dc87126e504611f19aceb391bf03f413
OFFICE SUMMARY = e4d84686504611f19aceb391bf03f413
DEPARTMENT KB = 0ccc2e8e333f11f18a174f38ce4f3987
DEPARTMENT SUMMARY = d9ef828233dd11f18a174f38ce4f3987
GLOBAL_CONCURRENCY = 100
RAGFLOW_CONCURRENCY = 30


## ragflow_client.py测试

In [ ]:
import asyncio
import httpx

from app.core.config import settings
from app.schemas.chat import ChatType
from app.services.ragflow_client import RagflowClient
from app.services.user_service import UserGroup


async def main():
    timeout = httpx.Timeout(
        connect=settings.RAGFLOW_CONNECT_TIMEOUT,
        read=settings.RAGFLOW_READ_TIMEOUT,
        write=settings.RAGFLOW_WRITE_TIMEOUT,
        pool=settings.RAGFLOW_POOL_TIMEOUT,
    )

    limits = httpx.Limits(
        max_connections=settings.HTTP_MAX_CONNECTIONS,
        max_keepalive_connections=settings.HTTP_MAX_KEEPALIVE_CONNECTIONS,
    )

    async with httpx.AsyncClient(timeout=timeout, limits=limits) as client:
        ragflow_client = RagflowClient(client)

        user_group = UserGroup.OFFICE_LEADER
        chat_type = ChatType.KB

        print("=" * 80)
        print("测试用户分组:", user_group)
        print("测试功能类型:", chat_type)

        chat_id = ragflow_client.get_chat_id(
            user_group=user_group,
            chat_type=chat_type,
        )
        print("匹配到的 chat_id:", chat_id)

        print("=" * 80)
        print("开始创建 session")

        session_id = await ragflow_client.create_session(
            user_group=user_group,
            chat_type=chat_type,
            session_name="StreamGate 测试会话",
        )

        print("创建成功，session_id:", session_id)

        print("=" * 80)
        print("开始流式问答")
        print()

        async for event in ragflow_client.stream_chat(
            user_group=user_group,
            chat_type=chat_type,
            session_id=session_id,
            question="城市更新主要任务是什么？",
        ):
            event_type = event.get("type")

            if event_type == "answer":
                print(event.get("answer", ""), flush=True)

            elif event_type == "reference":
                print()
                print("[命中知识库 reference]")
                reference = event.get("reference", {})
                chunks = reference.get("chunks", [])

                for chunk in chunks:
                    print(
                        "-",
                        chunk.get("document_name"),
                        "similarity:",
                        chunk.get("similarity"),
                    )

            elif event_type == "done":
                print()
                print("[DONE]")

        print("=" * 80)
        print("测试结束")


# if __name__ == "__main__":
#     asyncio.run(main())

In [4]:
await main()

测试用户分组: UserGroup.OFFICE_LEADER
测试功能类型: ChatType.KB
匹配到的 chat_id: dc87126e504611f19aceb391bf03f413
开始创建 session
创建成功，session_id: 818a2e8a51d811f1a5be253d403d7ec5
开始流式问答

好，我现在需要回答用户的问题：“城市更新

[命中知识库 reference]
主要任务是什么？”首先，我要仔细

[命中知识库 reference]
查看知识库中的相关内容，找出与之相关

[命中知识库 reference]
的信息。

浏览知识库，发现ID:

[命中知识库 reference]
0的内容提到了中共中央办公厅

[命中知识库 reference]
关于持续推进城市更新行动的意见，其中

[命中知识库 reference]
详细列出了八项主要任务。这些任务

[命中知识库 reference]
包括加强既有建筑改造利用

[命中知识库 reference]
、推进城镇老旧小区整治改造、

[命中知识库 reference]
开展完整社区建设、更新老旧街区

[命中知识库 reference]
、老旧厂区、城中村等，完善

[命中知识库 reference]
城市功能，加强基础设施建设，

[命中知识库 reference]
修复生态系统，保护历史文化。这些

[命中知识库 reference]
内容正好回答了用户的问题。

接着，ID

[命中知识库 reference]
:1和ID:2的内容主要是关于江西省

[命中知识库 reference]
的城市更新规划编制指南，虽然也

[命中知识库 reference]
提到了更新专项和片区，但没有具体的任务

[命中知识库 reference]
列表，因此与用户的问题关联性不大。

ID:

[命中知识库 reference]
3的内容与ID:0类似，同样提到了

[命中知识库 reference]
八项主要任务，但可能是重复的信息。

ID

[命中知识库 reference]
:4提到了实施评估、规划修改

[命中知识库 reference]
和档案移交，这些属于规划的

[命中知识库 reference]
实施和管理部分，和主要任务无关。

因此

[命中知识库

# Fastapi测试

### Session创建测试

In [44]:
async def create_session_only(
    client: httpx.AsyncClient,
    idx: int,
    user_group: UserGroup,
    chat_type: ChatType,
):
    ragflow_client = RagflowClient(client)

    start = time.perf_counter()

    session_id = await ragflow_client.create_session(
        user_group=user_group,
        chat_type=chat_type,
        session_name=f"session_only_{idx}_{int(time.time() * 1000)}",
    )

    elapsed = time.perf_counter() - start

    return {
        "idx": idx,
        "session_id": session_id,
        "session_create_time": elapsed,
    }

In [45]:
async def bench_create_session_only(concurrency: int = 10):
    timeout = httpx.Timeout(
        connect=10.0,
        read=120.0,
        write=10.0,
        pool=30.0,
    )

    limits = httpx.Limits(
        max_connections=concurrency + 10,
        max_keepalive_connections=concurrency + 10,
    )

    async with httpx.AsyncClient(timeout=timeout, limits=limits) as client:
        tasks = [
            create_session_only(
                client=client,
                idx=i + 1,
                user_group=UserGroup.OFFICE_LEADER,
                chat_type=ChatType.KB,
            )
            for i in range(concurrency)
        ]

        results = await asyncio.gather(*tasks)

    return pd.DataFrame(results)

In [46]:
session_df = await bench_create_session_only(10)
display(session_df)

,idx,session_id,session_create_time
0,1,7594a7c851f511f1a5be253d403d7ec5,0.210461
1,2,7596227451f511f1a5be253d403d7ec5,0.214871
2,3,7596e36251f511f1a5be253d403d7ec5,0.216975
3,4,75bb750651f511f1a5be253d403d7ec5,0.703726
4,5,75a239c451f511f1a5be253d403d7ec5,0.290954
5,6,762da73e51f511f1a5be253d403d7ec5,1.199911
6,7,7595b95651f511f1a5be253d403d7ec5,0.205594
7,8,75bcdd7e51f511f1a5be253d403d7ec5,0.456083
8,9,75c557d851f511f1a5be253d403d7ec5,0.512282
9,10,759c5ffe51f511f1a5be253d403d7ec5,0.240896


In [47]:
session_df = await bench_create_session_only(5)
display(session_df)

,idx,session_id,session_create_time
0,1,81a2851c51f511f1a5be253d403d7ec5,0.472178
1,2,817cc2dc51f511f1a5be253d403d7ec5,0.220860
2,3,81a0d46051f511f1a5be253d403d7ec5,0.460683
3,4,817bfa6e51f511f1a5be253d403d7ec5,0.213430
4,5,82612c0651f511f1a5be253d403d7ec5,1.997478


In [48]:
session_df = await bench_create_session_only(6)
display(session_df)

,idx,session_id,session_create_time
0,1,85512f4c51f511f1a5be253d403d7ec5,0.242591
1,2,854fd76e51f511f1a5be253d403d7ec5,0.182873
2,3,857a5ee451f511f1a5be253d403d7ec5,0.461653
3,4,85509b4051f511f1a5be253d403d7ec5,0.183889
4,5,85a284fa51f511f1a5be253d403d7ec5,0.723401
5,6,85550c0251f511f1a5be253d403d7ec5,0.209917


### 流式对话测试

In [13]:
import json
import time
import requests


STREAMGATE_URL = "http://127.0.0.1:19000/api/chat/stream"


def test_streamgate_stream():
    headers = {
        "Content-Type": "application/json",
        # 开发阶段临时用户身份
        # office_test -> 处室领导
        # department_test -> 厅领导
        "X-User-Id": "office_test",
    }

    payload = {
        "chat_type": "kb",
        "question": "城市更新主要任务是什么？",
        # 新建对话时不传 session_id
        # "session_id": "已有_session_id",
        "session_name": "StreamGate API 测试会话",
    }

    print("=" * 80)
    print("请求地址:", STREAMGATE_URL)
    print("请求头:", headers)
    print("请求体:")
    print(json.dumps(payload, ensure_ascii=False, indent=2))
    print("=" * 80)
    print("流式返回:")
    print()

    start_time = time.perf_counter()
    first_event_time = None

    with requests.post(
        STREAMGATE_URL,
        headers=headers,
        json=payload,
        stream=True,
        timeout=(10, 300),
    ) as response:
        print("HTTP 状态码:", response.status_code)
        print("=" * 80)

        if response.status_code != 200:
            print(response.text)
            return

        for line in response.iter_lines(decode_unicode=True):
            if not line:
                continue

            if line.startswith("data:"):
                raw = line[len("data:"):].strip()
            else:
                raw = line.strip()

            if not raw:
                continue

            if raw == "[DONE]":
                print()
                print("[DONE]")
                break

            if first_event_time is None:
                first_event_time = time.perf_counter()
                print(f"[首个事件延迟: {first_event_time - start_time:.3f}s]")
                print("-" * 80)

            try:
                event = json.loads(raw)
            except json.JSONDecodeError:
                print("[非 JSON]:", raw)
                continue

            event_type = event.get("type")

            if event_type == "start":
                print("[start]", event.get("request_id"))

            elif event_type == "session":
                print("[session_id]", event.get("session_id"))

            elif event_type == "answer":
                print(event.get("answer", ""), flush=True)

            elif event_type == "reference":
                print()
                print("[reference]")
                reference = event.get("reference", {})
                chunks = reference.get("chunks", [])
                for chunk in chunks:
                    print(
                        "-",
                        chunk.get("document_name"),
                        "similarity:",
                        chunk.get("similarity"),
                    )

            elif event_type == "done":
                print()
                print("[ragflow done]")

            elif event_type == "end":
                print()
                print("[end]", event)

            elif event_type == "error":
                print()
                print("[error]", event)

            else:
                print("[event]", event)

    total_time = time.perf_counter() - start_time

    print()
    print("=" * 80)
    print(f"总耗时: {total_time:.3f}s")
    print("=" * 80)



In [14]:

# if __name__ == "__main__":
test_streamgate_stream()

请求地址: http://127.0.0.1:19000/api/chat/stream
请求头: {'Content-Type': 'application/json', 'X-User-Id': 'office_test'}
请求体:
{
  "chat_type": "kb",
  "question": "城市更新主要任务是什么？",
  "session_name": "StreamGate API 测试会话"
}
流式返回:

HTTP 状态码: 200
[首个事件延迟: 0.005s]
--------------------------------------------------------------------------------
[start] 7d1cf8bb805440e6825ad23178d6df1c
[session_id] 6a67558651ef11f1a5be253d403d7ec5
好，我现在要回答用户的问题：“城市更新

[reference]
主要任务是什么？”首先，我需要仔细

[reference]
查看知识库中的内容，找出相关的信息。



[reference]
知识库中有几个文档提到了城市

[reference]
更新的主要任务。ID:0的文档内容中提

[reference]
到，意见部署了八项主要任务，包括

[reference]
加强既有建筑改造利用、推进

[reference]
城镇老旧小区整治改造、开展完整

[reference]
社区建设、推进老旧街区、老旧

[reference]
厂区、城中村等更新改造、完善

[reference]
城市功能、加强城市基础设施建设

[reference]
改造、修复城市生态系统、保护传承

[reference]
城市历史文化。这些内容在ID:0的

[reference]
正文部分有详细说明。

另外，ID:

[reference]
3的文档内容也提到了同样的八项

[reference]
任务，并且每个任务都有具体的描述和背景

[reference]
信息。例如，切片ID:3详细说明了每

[reference]
个任务的具体内容和目的，引用了相关专家的

[reference]
观点来支持这些任务的实施。

此外

[referenc

## 并发

In [38]:
import asyncio
import json
import time
from dataclasses import dataclass
from statistics import mean, median
from typing import Optional

import httpx
import pandas as pd
from IPython.display import display
import pandas as pd
from statistics import mean
from IPython.display import display

STREAMGATE_URL = "http://127.0.0.1:19000/api/chat/stream"

# 测试用户：
# office_test -> 处室领导
# department_test -> 厅领导
USER_ID = "office_test"

# 测试功能：
# kb      -> 知识库问答
# summary -> 文件总结
CHAT_TYPE = "kb"

# 每轮并发数
CONCURRENCY_LIST = [5, 10, 15]

# 每个请求的问题列表。
# 如果问题数量少于并发数，会循环取用。
QUESTIONS = [
    "城市更新主要任务是什么？",
    "请说明城市更新的实施路径。",
    "城市更新中如何统筹历史文化保护？",
    "城市更新和城中村改造有什么关系？",
    "城市更新项目推进中有哪些重点环节？",
    "请总结城市更新的政策重点。",
    "城市更新如何与产业导入结合？",
    "城市更新中如何保障公共利益？",
    "城市更新中的存量空间盘活怎么理解？",
    "请从规划管理角度说明城市更新重点。",
    "城市更新对土地节约集约利用有什么作用？",
    "城市更新中政府、市场、居民如何协同？",
    "城市更新如何提升城市品质？",
    "城市更新项目有哪些常见难点？",
    "请用条目化方式回答城市更新主要任务。",
]


@dataclass
class RequestResult:
    idx: int
    question: str
    mode: str
    success: bool
    status_code: Optional[int]

    session_create_time: Optional[float]
    first_answer_after_completion: Optional[float]
    ttft: Optional[float]
    generation_time: Optional[float]
    total_time: float

    output_chars: int
    speed_chars_per_sec: Optional[float]
    session_id: Optional[str]
    preview: str
    error: Optional[str]

def percentile(values: list[float], p: float) -> Optional[float]:
    """
    简单计算百分位数。
    p 取值：0.5 / 0.9 / 0.95
    """
    if not values:
        return None

    values = sorted(values)
    k = int(round((len(values) - 1) * p))
    return values[k]


def format_float(value: Optional[float]) -> str:
    if value is None:
        return "-"
    return f"{value:.3f}"


def shorten(value: object, max_len: int = 60) -> str:
    """
    截断长文本，避免表格太宽。
    """
    if value is None:
        return "-"

    text = str(value)

    if len(text) <= max_len:
        return text

    return text[: max_len - 3] + "..."


async def run_one_request(
    client: httpx.AsyncClient,
    idx: int,
    question: str,
    mode: str = "new_session",
    session_id: Optional[str] = None,
) -> RequestResult:
    headers = {
        "Content-Type": "application/json",
        "X-User-Id": USER_ID,
    }

    payload = {
        "chat_type": CHAT_TYPE,
        "question": question,
    }

    if mode == "new_session":
        payload["session_name"] = f"bench_{CHAT_TYPE}_{idx}_{int(time.time() * 1000)}"
    elif mode == "existing_session":
        if not session_id:
            raise ValueError("existing_session 模式必须传 session_id")
        payload["session_id"] = session_id
    else:
        raise ValueError(f"不支持的测试模式: {mode}")

    request_start = time.perf_counter()
    completion_start: Optional[float] = None
    first_answer_time: Optional[float] = None

    session_create_time: Optional[float] = None
    returned_session_id: Optional[str] = None
    output_parts: list[str] = []
    status_code: Optional[int] = None

    try:
        async with client.stream(
            "POST",
            STREAMGATE_URL,
            headers=headers,
            json=payload,
        ) as response:
            status_code = response.status_code

            if response.status_code != 200:
                text = await response.aread()
                total_time = time.perf_counter() - request_start
                return RequestResult(
                    idx=idx,
                    question=question,
                    mode=mode,
                    success=False,
                    status_code=status_code,
                    session_create_time=session_create_time,
                    first_answer_after_completion=None,
                    ttft=None,
                    generation_time=None,
                    total_time=total_time,
                    output_chars=0,
                    speed_chars_per_sec=None,
                    session_id=None,
                    preview="",
                    error=f"HTTP {status_code}: {text.decode('utf-8', errors='ignore')[:500]}",
                )

            async for line in response.aiter_lines():
                if not line:
                    continue

                if line.startswith("data:"):
                    raw = line[len("data:"):].strip()
                else:
                    raw = line.strip()

                if not raw:
                    continue

                if raw == "[DONE]":
                    break

                try:
                    event = json.loads(raw)
                except json.JSONDecodeError:
                    continue

                event_type = event.get("type")

                if event_type == "session":
                    returned_session_id = event.get("session_id")
                    session_create_time = event.get("session_create_time")
                    completion_start = time.perf_counter()

                elif event_type == "answer":
                    answer = event.get("answer", "")

                    if answer:
                        if first_answer_time is None:
                            first_answer_time = time.perf_counter()

                        current_text = "".join(output_parts)

                        # 兼容累计 answer 和增量 answer
                        if answer.startswith(current_text):
                            delta = answer[len(current_text):]
                            output_parts.append(delta)
                        else:
                            output_parts.append(answer)

                elif event_type == "error":
                    total_time = time.perf_counter() - request_start
                    current_output = "".join(output_parts)

                    ttft = (
                        first_answer_time - request_start
                        if first_answer_time is not None
                        else None
                    )

                    first_answer_after_completion = (
                        first_answer_time - completion_start
                        if first_answer_time is not None and completion_start is not None
                        else None
                    )

                    generation_time = (
                        total_time - ttft
                        if ttft is not None
                        else None
                    )

                    return RequestResult(
                        idx=idx,
                        question=question,
                        mode=mode,
                        success=False,
                        status_code=status_code,
                        session_create_time=session_create_time,
                        first_answer_after_completion=first_answer_after_completion,
                        ttft=ttft,
                        generation_time=generation_time,
                        total_time=total_time,
                        output_chars=len(current_output),
                        speed_chars_per_sec=None,
                        session_id=returned_session_id,
                        preview=current_output[:160],
                        error=json.dumps(event, ensure_ascii=False),
                    )

                elif event_type == "done":
                    break

    except Exception as exc:
        total_time = time.perf_counter() - request_start
        current_output = "".join(output_parts)

        ttft = (
            first_answer_time - request_start
            if first_answer_time is not None
            else None
        )

        first_answer_after_completion = (
            first_answer_time - completion_start
            if first_answer_time is not None and completion_start is not None
            else None
        )

        generation_time = (
            total_time - ttft
            if ttft is not None
            else None
        )

        return RequestResult(
            idx=idx,
            question=question,
            mode=mode,
            success=False,
            status_code=status_code,
            session_create_time=session_create_time,
            first_answer_after_completion=first_answer_after_completion,
            ttft=ttft,
            generation_time=generation_time,
            total_time=total_time,
            output_chars=len(current_output),
            speed_chars_per_sec=None,
            session_id=returned_session_id,
            preview=current_output[:160],
            error=repr(exc),
        )

    total_time = time.perf_counter() - request_start
    final_answer = "".join(output_parts)
    output_chars = len(final_answer)

    ttft = (
        first_answer_time - request_start
        if first_answer_time is not None
        else None
    )

    first_answer_after_completion = (
        first_answer_time - completion_start
        if first_answer_time is not None and completion_start is not None
        else None
    )

    generation_time = (
        total_time - ttft
        if ttft is not None
        else None
    )

    if generation_time is not None and generation_time > 0:
        speed = output_chars / generation_time
    elif total_time > 0:
        speed = output_chars / total_time
    else:
        speed = None

    return RequestResult(
        idx=idx,
        question=question,
        mode=mode,
        success=True,
        status_code=status_code,
        session_create_time=session_create_time,
        first_answer_after_completion=first_answer_after_completion,
        ttft=ttft,
        generation_time=generation_time,
        total_time=total_time,
        output_chars=output_chars,
        speed_chars_per_sec=speed,
        session_id=returned_session_id,
        preview=final_answer[:160],
        error=None,
    )


async def run_concurrency(
    concurrency: int,
    mode: str = "new_session",
    session_ids: Optional[list[str]] = None,
) -> list[RequestResult]:
    timeout = httpx.Timeout(
        connect=10.0,
        read=300.0,
        write=10.0,
        pool=30.0,
    )

    limits = httpx.Limits(
        max_connections=concurrency + 10,
        max_keepalive_connections=concurrency + 10,
    )

    async with httpx.AsyncClient(timeout=timeout, limits=limits) as client:
        tasks = []

        for i in range(concurrency):
            question = QUESTIONS[i % len(QUESTIONS)]

            sid = None
            if mode == "existing_session":
                if not session_ids or i >= len(session_ids):
                    raise ValueError("existing_session 模式下 session_ids 数量不足")
                sid = session_ids[i]

            tasks.append(
                run_one_request(
                    client=client,
                    idx=i + 1,
                    question=question,
                    mode=mode,
                    session_id=sid,
                )
            )

        results = await asyncio.gather(*tasks)
        return results


def print_table(headers: list[str], rows: list[list[object]]) -> None:
    """
    打印简单文本表格。
    """
    str_rows = [[str(cell) for cell in row] for row in rows]
    str_headers = [str(h) for h in headers]

    widths = []
    for i, header in enumerate(str_headers):
        max_width = len(header)
        for row in str_rows:
            if i < len(row):
                max_width = max(max_width, len(row[i]))
        widths.append(max_width)

    def fmt_row(row: list[str]) -> str:
        return " | ".join(
            cell.ljust(widths[i])
            for i, cell in enumerate(row)
        )

    sep = "-+-".join("-" * w for w in widths)

    print(fmt_row(str_headers))
    print(sep)

    for row in str_rows:
        print(fmt_row(row))


import pandas as pd
from statistics import mean
from IPython.display import display


def build_summary_dataframes(
    concurrency: int,
    results: list[RequestResult],
    wall_time: float,
):
    success_results = [r for r in results if r.success]
    failed_results = [r for r in results if not r.success]

    ttfts = [r.ttft for r in success_results if r.ttft is not None]
    total_times = [r.total_time for r in success_results]
    speeds = [
        r.speed_chars_per_sec
        for r in success_results
        if r.speed_chars_per_sec is not None
    ]
    output_chars = [r.output_chars for r in success_results]
    
    session_create_times = [
        r.session_create_time
        for r in success_results
        if r.session_create_time is not None
    ]

    first_answer_after_completion_times = [
        r.first_answer_after_completion
        for r in success_results
        if r.first_answer_after_completion is not None
    ]

    generation_times = [
        r.generation_time
        for r in success_results
        if r.generation_time is not None
    ]
    total_output_chars = sum(output_chars)
    overall_throughput = total_output_chars / wall_time if wall_time > 0 else None

    overview_df = pd.DataFrame(
        [
            ["并发数", concurrency],
            ["总耗时/秒", round(wall_time, 3)],
            ["请求总数", len(results)],
            ["成功数", len(success_results)],
            ["失败数", len(failed_results)],
            ["成功率", f"{len(success_results) / len(results) * 100:.1f}%" if results else "-"],
            ["总输出字符数", total_output_chars],
            ["整体吞吐/字符每秒", round(overall_throughput, 3) if overall_throughput is not None else "-"],
        ],
        columns=["指标", "数值"],
    )

    metrics_df = pd.DataFrame(
        [
            [
                "Session创建耗时/秒",
                round(mean(session_create_times), 3) if session_create_times else None,
                round(percentile(session_create_times, 0.50), 3) if session_create_times else None,
                round(percentile(session_create_times, 0.90), 3) if session_create_times else None,
                round(percentile(session_create_times, 0.95), 3) if session_create_times else None,
            ],
            [
                "Completions首答耗时/秒",
                round(mean(first_answer_after_completion_times), 3) if first_answer_after_completion_times else None,
                round(percentile(first_answer_after_completion_times, 0.50), 3) if first_answer_after_completion_times else None,
                round(percentile(first_answer_after_completion_times, 0.90), 3) if first_answer_after_completion_times else None,
                round(percentile(first_answer_after_completion_times, 0.95), 3) if first_answer_after_completion_times else None,
            ],
            [
                "TTFT总首字延迟/秒",
                round(mean(ttfts), 3) if ttfts else None,
                round(percentile(ttfts, 0.50), 3) if ttfts else None,
                round(percentile(ttfts, 0.90), 3) if ttfts else None,
                round(percentile(ttfts, 0.95), 3) if ttfts else None,
            ],
            [
                "生成阶段耗时/秒",
                round(mean(generation_times), 3) if generation_times else None,
                round(percentile(generation_times, 0.50), 3) if generation_times else None,
                round(percentile(generation_times, 0.90), 3) if generation_times else None,
                round(percentile(generation_times, 0.95), 3) if generation_times else None,
            ],
            [
                "总耗时/秒",
                round(mean(total_times), 3) if total_times else None,
                round(percentile(total_times, 0.50), 3) if total_times else None,
                round(percentile(total_times, 0.90), 3) if total_times else None,
                round(percentile(total_times, 0.95), 3) if total_times else None,
            ],
            [
                "输出速度/字符每秒",
                round(mean(speeds), 3) if speeds else None,
                round(percentile(speeds, 0.50), 3) if speeds else None,
                round(percentile(speeds, 0.90), 3) if speeds else None,
                round(percentile(speeds, 0.95), 3) if speeds else None,
            ],
            [
                "输出字符数",
                round(mean(output_chars), 3) if output_chars else None,
                round(percentile(output_chars, 0.50), 3) if output_chars else None,
                round(percentile(output_chars, 0.90), 3) if output_chars else None,
                round(percentile(output_chars, 0.95), 3) if output_chars else None,
            ],
        ],
        columns=["指标", "平均值", "P50", "P90", "P95"],
    )

    details_df = pd.DataFrame(
        [
            {
                "序号": r.idx,
                "模式": r.mode,
                "成功": r.success,
                "状态码": r.status_code,
                "Session创建/s": round(r.session_create_time, 3) if r.session_create_time is not None else None,
                "首答耗时/s": round(r.first_answer_after_completion, 3) if r.first_answer_after_completion is not None else None,
                "TTFT/s": round(r.ttft, 3) if r.ttft is not None else None,
                "生成耗时/s": round(r.generation_time, 3) if r.generation_time is not None else None,
                "总耗时/s": round(r.total_time, 3),
                "输出字符数": r.output_chars,
                "速度/字符每秒": round(r.speed_chars_per_sec, 3) if r.speed_chars_per_sec is not None else None,
                "Session": r.session_id,
                "问题": r.question,
            }
            for r in results
        ]
    )

    preview_df = pd.DataFrame(
        [
            {
                "序号": r.idx,
                "问题": r.question,
                "回答预览": r.preview.replace("\n", " ")[:200] if r.preview else "",
            }
            for r in results
        ]
    )

    errors_df = pd.DataFrame(
        [
            {
                "序号": r.idx,
                "状态码": r.status_code,
                "错误": r.error,
            }
            for r in failed_results
        ]
    )

    return overview_df, metrics_df, details_df, preview_df, errors_df


async def main():
    all_overview = []
    all_metrics = []
    all_details = []
    all_preview = []
    all_errors = []

    for concurrency in CONCURRENCY_LIST:
        print(f"开始测试并发数: {concurrency}")

        start = time.perf_counter()
        results = await run_concurrency(concurrency)
        wall_time = time.perf_counter() - start

        overview_df, metrics_df, details_df, preview_df, errors_df = build_summary_dataframes(
            concurrency=concurrency,
            results=results,
            wall_time=wall_time,
        )

        # 给每张表加上并发数字段，方便后面合并比较
        overview_df.insert(0, "并发数", concurrency)
        metrics_df.insert(0, "并发数", concurrency)
        details_df.insert(0, "并发数", concurrency)
        preview_df.insert(0, "并发数", concurrency)

        all_overview.append(overview_df)
        all_metrics.append(metrics_df)
        all_details.append(details_df)
        all_preview.append(preview_df)

        if not errors_df.empty:
            errors_df.insert(0, "并发数", concurrency)
            all_errors.append(errors_df)

    overview_all_df = pd.concat(all_overview, ignore_index=True)
    metrics_all_df = pd.concat(all_metrics, ignore_index=True)
    details_all_df = pd.concat(all_details, ignore_index=True)
    preview_all_df = pd.concat(all_preview, ignore_index=True)

    print("一、总体情况")
    display(overview_all_df)

    print("二、性能指标")
    display(metrics_all_df)

    print("三、单请求明细")
    display(details_all_df)

    print("四、回答预览")
    display(preview_all_df)

    if all_errors:
        errors_all_df = pd.concat(all_errors, ignore_index=True)
        print("五、失败详情")
        display(errors_all_df)

    return {
        "overview": overview_all_df,
        "metrics": metrics_all_df,
        "details": details_all_df,
        "preview": preview_all_df,
        "errors": pd.concat(all_errors, ignore_index=True) if all_errors else pd.DataFrame(),
    }


# if __name__ == "__main__":
#     asyncio.run(main())

In [39]:
STREAMGATE_URL = "http://127.0.0.1:19000/api/chat/stream"

# 依次测试这些并发数
CONCURRENCY_LIST = [10]

# 每个并发档位的问题列表
# 如果问题数量少于并发数，会自动循环使用
QUESTIONS = [
    "城市更新主要任务是什么？",
    "请说明城市体检和城市更新的关系。",
    "老旧小区改造的重点内容有哪些？",
    "城市更新中如何推进历史文化保护？",
    "城市更新项目实施路径有哪些？",
    "请总结城市更新工作的政策重点。",
    "城市更新如何与十五分钟生活圈结合？",
    "城市更新中如何加强公众参与？",
    "城市更新和存量用地盘活有什么关系？",
    "请说明城市更新中的资金保障机制。",
    "城市更新如何支撑高质量发展？",
    "城市更新中如何处理产业导入问题？",
    "城市更新如何提升公共服务水平？",
    "城市更新中如何推进空间品质提升？",
    "城市更新实施过程中有哪些风险？",
]


### 新建session

In [41]:
results = await run_concurrency(10, mode="new_session")

In [42]:
results

[RequestResult(idx=1, question='城市更新主要任务是什么？', mode='new_session', success=True, status_code=200, session_create_time=0.236, first_answer_after_completion=3.42424399999436, ttft=3.7313541999901645, generation_time=42.371791400015354, total_time=46.10314560000552, output_chars=629, speed_chars_per_sec=14.844781851724402, session_id='00dcf38251f411f1a5be253d403d7ec5', preview='好的，我现在来思考一下用户的问题：“城市更新主要任务是什么？”首先，我需要从知识库中找到相关的信息来回答这个问题。知识库中有多个文档提到了城市更新的主要任务，我需要仔细查看这些内容。\n\n从ID:0的内容来看，中共中央办公厅和国务院办公厅提出了八项主要任务，包括加强既有建筑改造利用、推进城镇老旧小区整治改造、开展完整社区', error=None),
 RequestResult(idx=2, question='请说明城市体检和城市更新的关系。', mode='new_session', success=True, status_code=200, session_create_time=1.799, first_answer_after_completion=3.3257090999977663, ttft=5.165187400009017, generation_time=91.38112169998931, total_time=96.54630909999833, output_chars=1820, speed_chars_per_sec=19.916586337987717, session_id='01c46b1851f411f1a5be253d403d7ec5', preview='嗯，我现在要回答用户的问题：“请说明城市体检和城市更新的关系。”首先，我需要理解这两个概念的含义以及它们在深圳市的具体实践

### 复用session

In [ ]:
session_ids = [
    "session_1",
    "session_2",
    ...
]

### 同问题

In [ ]:
QUESTIONS = ["城市更新主要任务是什么？"] * 15

In [43]:
dfs = await main()

开始测试并发数: 10
一、总体情况


,并发数,指标,数值
0,10,并发数,10
1,10,总耗时/秒,83.956
2,10,请求总数,10
3,10,成功数,10
4,10,失败数,0
5,10,成功率,100.0%
6,10,总输出字符数,13165
7,10,整体吞吐/字符每秒,156.807


二、性能指标


,并发数,指标,平均值,P50,P90,P95
0,10,Session创建耗时/秒,1.196,0.361,2.159,2.201
1,10,Completions首答耗时/秒,3.301,2.925,3.806,3.812
2,10,TTFT总首字延迟/秒,4.567,4.131,5.108,5.109
3,10,生成阶段耗时/秒,67.413,61.863,77.757,78.895
4,10,总耗时/秒,71.980,65.987,81.888,83.925
5,10,输出速度/字符每秒,19.534,19.304,20.015,20.480
6,10,输出字符数,1316.500,1219.000,1543.000,1558.000


三、单请求明细


,并发数,序号,模式,成功,状态码,Session创建/s,首答耗时/s,TTFT/s,生成耗时/s,总耗时/s,输出字符数,速度/字符每秒,Session,问题
0,10,1,new_session,True,200,0.238,3.484,3.795,58.591,62.386,1168,19.935,856ff52251f411f1a5be253d403d7ec5,城市更新主要任务是什么？
1,10,2,new_session,True,200,0.234,3.812,4.124,61.863,65.987,1219,19.705,8570932451f411f1a5be253d403d7ec5,请说明城市体检和城市更新的关系。
2,10,3,new_session,True,200,0.198,3.788,4.107,77.071,81.178,1459,18.931,8572710851f411f1a5be253d403d7ec5,老旧小区改造的重点内容有哪些？
3,10,4,new_session,True,200,0.231,3.806,4.126,77.203,81.329,1543,19.986,8573199651f411f1a5be253d403d7ec5,城市更新中如何推进历史文化保护？
4,10,5,new_session,True,200,2.159,2.918,5.109,76.073,81.182,1558,20.480,868f407051f411f1a5be253d403d7ec5,城市更新项目实施路径有哪些？
5,10,6,new_session,True,200,0.361,3.727,4.131,77.757,81.888,1501,19.304,857f8f8c51f411f1a5be253d403d7ec5,请总结城市更新工作的政策重点。
6,10,7,new_session,True,200,2.138,2.925,5.108,51.250,56.358,987,19.259,868e7a0a51f411f1a5be253d403d7ec5,城市更新如何与十五分钟生活圈结合？
7,10,8,new_session,True,200,2.063,2.858,5.036,55.324,60.360,1055,19.070,868de2a251f411f1a5be253d403d7ec5,城市更新中如何加强公众参与？
8,10,9,new_session,True,200,2.201,2.776,5.031,78.895,83.925,1472,18.658,868fd57651f411f1a5be253d403d7ec5,城市更新和存量用地盘活有什么关系？
9,10,10,new_session,True,200,2.142,2.917,5.103,60.104,65.207,1203,20.015,86905e1051f411f1a5be253d403d7ec5,请说明城市更新中的资金保障机制。


四、回答预览


,并发数,序号,问题,回答预览
0,10,1,城市更新主要任务是什么？,好的，我现在要回答用户的问题：“城市更新主要任务是什么？”首先，我需要仔细查看知识库中的相关...
1,10,2,请说明城市体检和城市更新的关系。,好吧，用户问的是城市体检和城市更新的关系。我得先看看知识库里的内容，里面有个ID:0的文件，...
2,10,3,老旧小区改造的重点内容有哪些？,嗯，用户问的是老旧小区改造的重点内容有哪些。首先，我得看看知识库里的相关内容。知识库里有几个...
3,10,4,城市更新中如何推进历史文化保护？,好，我现在需要回答用户的问题：“城市更新中如何推进历史文化保护？”首先，我要仔细查看提供的知...
4,10,5,城市更新项目实施路径有哪些？,好，我现在要回答用户的问题：“城市更新项目实施路径有哪些？”首先，我需要仔细查看提供的知识库...
5,10,6,请总结城市更新工作的政策重点。,好，我现在要帮用户总结城市更新工作的政策重点。首先，我需要仔细查看知识库中的四个文档，找出相...
6,10,7,城市更新如何与十五分钟生活圈结合？,好，我现在要回答用户的问题：“城市更新如何与十五分钟生活圈结合？”首先，我需要从知识库中找到...
7,10,8,城市更新中如何加强公众参与？,好的，我现在要回答用户的问题：“城市更新中如何加强公众参与？”首先，我需要查阅知识库中的相关...
8,10,9,城市更新和存量用地盘活有什么关系？,嗯，我现在要回答的问题是“城市更新和存量用地盘活有什么关系？”。首先，我需要理解这两个概念之...
9,10,10,请说明城市更新中的资金保障机制。,好的，我需要回答用户关于城市更新中资金保障机制的问题。首先，我查看了提供的知识库内容，特别是...


In [ ]:
dfs["metrics"]

In [ ]:
dfs["details"]